In [1]:
!pip -q install transformers accelerate pillow pandas scikit-learn tqdm sentencepiece

In [3]:
import re, json
from pathlib import Path
import pandas as pd
import torch
from tqdm import tqdm
from PIL import Image
from transformers import Blip2Processor, Blip2ForConditionalGeneration

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [4]:
from google.colab import drive
drive.mount('/content/drive')

# Paths (FIXED from previous experiments)
BASE_DIR = Path('/content/drive/MyDrive/colab_experiments')
DATA_DIR = BASE_DIR / 'data'
RESULTS_DIR = BASE_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load utilities by direct execution (bypass import issues)
print("Loading utilities...")

with open(BASE_DIR / 'utils/data_loader.py') as f:
    exec(f.read(), globals())

with open(BASE_DIR / 'utils/prompt_templates.py') as f:
    exec(f.read(), globals())

with open(BASE_DIR / 'utils/metrics.py') as f:
    exec(f.read(), globals())

print("✓ Utilities loaded successfully")


Mounted at /content/drive
Loading utilities...
✓ Utilities loaded successfully


In [5]:
# Load metadata with explanations
metadata_df = load_metadata(DATA_DIR / 'metadata.csv', DATA_DIR / 'explanations.csv')

print(f'✓ Loaded {len(metadata_df)} rows')
print(f'✓ Columns: {list(metadata_df.columns)}')

# Check explanation coverage
if 'explanation_implicit' in metadata_df.columns:
    has_explanation = (metadata_df['explanation_implicit'] != '') & (metadata_df['explanation_implicit'].notna())
    with_exp = has_explanation.sum()

    print(f'✓ With explanations: {with_exp}/650 ({with_exp/650*100:.1f}%)')
    print(f'✓ Without explanations: {650-with_exp}/650')

    if with_exp >= 580:
        print('\n✅ Ready for BLIP-2 inference!')

metadata_df.head(2)

✓ Loaded 650 rows
✓ Columns: ['image_id', 'category', 'ocr_text', 'human_label', 'image_path', 'explanation_implicit', 'human_label_vec']
✓ With explanations: 581/650 (89.4%)
✓ Without explanations: 69/650

✅ Ready for BLIP-2 inference!


,image_id,category,ocr_text,human_label,image_path,explanation_implicit,human_label_vec
0,TE-104.jpg,BRAND_DEPENDENT,इतनी tastyBiryani खाने के बादभी HEAL MY FEELINGS,"[1,0,0,0,0,0,0]",test_images/TE-104.jpg,CONTEXT: This is a bilingual (Hindi/English) I...,"[1, 0, 0, 0, 0, 0, 0]"
1,TE-177.jpg,BRAND_DEPENDENT,Porn hub 0 Previous 102 ^ 103| 101!* /05N 105|...,"[1,0,0,0,1,0,0]",test_images/TE-177.jpg,1. MOOD/EMOTIONAL STATE: The text uses a mocki...,"[1, 0, 0, 0, 1, 0, 0]"


In [6]:
model_id = 'Salesforce/blip2-flan-t5-xl'

print("Loading BLIP-2-Flan-T5-XL...")
print("This will download ~15GB (first time only)")
print("Expected time: 5-10 minutes\n")

processor = Blip2Processor.from_pretrained(model_id)
model = Blip2ForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,  # FP16 for A100
    device_map='auto'
)
model.eval()

print("\n✓ BLIP-2 loaded successfully!")
print(f"✓ Device: {next(model.parameters()).device}")

Loading BLIP-2-Flan-T5-XL...
This will download ~15GB (first time only)
Expected time: 5-10 minutes



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1289 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie language_model.shared.weight to language_model.lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]


✓ BLIP-2 loaded successfully!
✓ Device: cuda:0


In [7]:
def create_explanation_prompt(explanation_text):
    """Create prompt combining explanation with question"""
    if explanation_text and explanation_text.strip():
        return (
            f"Context: {explanation_text[:300]}\n\n"  # Use first 300 chars of explanation
            "Based on this meme, list all mental health issues shown. "
            "Options: depression, loss of interest, self-harm, eating problems, "
            "low self-esteem, concentration issues, sleep problems. "
            "List all that apply, separated by commas."
        )
    else:
        # Fallback for images without explanations
        return (
            "List all mental health issues shown in this meme. "
            "Options: depression, loss of interest, self-harm, eating problems, "
            "low self-esteem, concentration issues, sleep problems. "
            "List all that apply, separated by commas."
        )

def parse_symptom_list(response_text):
    """Convert natural language response to binary vector"""
    response_lower = response_text.lower()

    symptom_map = {
        'depression': 0,
        'loss of interest': 1,
        'self-harm': 2,
        'eating problems': 3,
        'low self-esteem': 4,
        'concentration issues': 5,
        'sleep problems': 6
    }

    vec = [0] * 7
    for symptom, idx in symptom_map.items():
        if symptom in response_lower:
            vec[idx] = 1

    return '[' + ','.join(str(x) for x in vec) + ']'

# Run inference
final_csv = RESULTS_DIR / 'blip2_explanation_augmented.csv'
checkpoint_interval = 50

print("=== BLIP-2 EXPLANATION-AUGMENTED INFERENCE ===")
print(f"Processing {len(metadata_df)} images with natural language prompts")
print("Using explanations when available (581/650)")
print("This will take ~5-10 minutes\n")

rows = []
for idx, row in tqdm(metadata_df.iterrows(), total=len(metadata_df)):
    try:
        image = Image.open(DATA_DIR / row['image_path']).convert('RGB')
        prompt = create_explanation_prompt(row.get('explanation_implicit', ''))

        inputs = processor(images=image, text=prompt, return_tensors='pt').to(model.device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=True,
                temperature=0.7,
                top_p=0.9
            )

        response = processor.batch_decode(out, skip_special_tokens=True)[0]
        pred_vec = parse_symptom_list(response)

        rec = {
            'image_id': row['image_id'],
            'category': row['category'],
            'prediction_raw': response,
            'prediction_vec': pred_vec,
            'had_explanation': bool(row.get('explanation_implicit', '').strip()),
            'human_label': row['human_label']
        }
    except Exception as e:
        rec = {
            'image_id': row['image_id'],
            'category': row['category'],
            'prediction_raw': f'ERROR: {e}',
            'prediction_vec': '[0,0,0,0,0,0,0]',
            'had_explanation': False,
            'human_label': row['human_label']
        }
    rows.append(rec)

    if len(rows) % checkpoint_interval == 0:
        pd.DataFrame(rows).to_csv(final_csv, index=False)
        print(f'✓ Checkpoint: {len(rows)} images')

res_df = pd.DataFrame(rows)
res_df.to_csv(final_csv, index=False)

print(f'\n🎉 Inference complete!')
print(f'✓ Saved: {final_csv}')
print(f'✓ Total images: {len(res_df)}')
print(f'✓ With explanations: {res_df["had_explanation"].sum()}')

=== BLIP-2 EXPLANATION-AUGMENTED INFERENCE ===
Processing 650 images with natural language prompts
Using explanations when available (581/650)
This will take ~5-10 minutes



  8%|▊         | 50/650 [02:20<44:48,  4.48s/it]

✓ Checkpoint: 50 images


 15%|█▌        | 100/650 [04:21<23:28,  2.56s/it]

✓ Checkpoint: 100 images


...

✓ Checkpoint: 550 images


 92%|█████████▏| 600/650 [25:07<01:22,  1.65s/it]

✓ Checkpoint: 600 images


100%|██████████| 650/650 [27:18<00:00,  2.52s/it]

✓ Checkpoint: 650 images

🎉 Inference complete!
✓ Saved: /content/drive/MyDrive/colab_experiments/results/blip2_explanation_augmented.csv
✓ Total images: 650
✓ With explanations: 581


In [8]:
# Load results and calculate metrics
res_df = pd.read_csv(RESULTS_DIR / 'blip2_explanation_augmented.csv')

# Parse predictions
y_true = parse_predictions(res_df['human_label'].tolist())
y_pred = parse_predictions(res_df['prediction_vec'].tolist())

# Calculate metrics
metrics = calculate_metrics(y_true, y_pred, SYMPTOM_NAMES)

print("\n" + "=" * 80)
print("=== BLIP-2 EXPLANATION-AUGMENTED RESULTS ===")
print("=" * 80)
print_results_table(metrics)

# Save metrics
save_metrics_csv(metrics, RESULTS_DIR / 'blip2_explanation_augmented_metrics.csv')

import json
with open(RESULTS_DIR / 'blip2_explanation_augmented_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("\n✓ Metrics saved!")

# Compare to baseline
print("\n" + "=" * 80)
print("=== COMPARISON TO BASELINE ===")
print("=" * 80)
print(f"InstructBLIP (zero-shot):           Macro-F1 = 0.25 (25%)")
print(f"BLIP-2 (explanation-augmented):     Macro-F1 = {metrics['macro_f1']:.4f} ({metrics['macro_f1']*100:.1f}%)")

improvement = (metrics['macro_f1'] - 0.25) / 0.25 * 100
print(f"\nImprovement: {improvement:+.1f}% relative to baseline")

# Show sample predictions
print("\n" + "=" * 80)
print("=== SAMPLE PREDICTIONS ===")
print("=" * 80)
for i in range(5):
    row = res_df.iloc[i]
    print(f"\nImage: {row['image_id']}")
    print(f"Had explanation: {row['had_explanation']}")
    print(f"Model said: '{row['prediction_raw']}'")
    print(f"Predicted: {row['prediction_vec']}")
    print(f"Actual:    {row['human_label']}")


=== BLIP-2 EXPLANATION-AUGMENTED RESULTS ===
n=650
Macro-F1=0.0126 | Weighted-F1=0.0270 | Macro-P=0.0449 | Macro-R=0.0073
--------------------------------------------------------------------------------
Symptom                              P       R      F1    Gold+    Pred+
--------------------------------------------------------------------------------
Feeling Down                     0.314   0.051   0.088      215       35
Lack of Interest                 0.000   0.000   0.000       65        0
Self-Harm                        0.000   0.000   0.000       79        0
Eating Disorder                  0.000   0.000   0.000       82        0
Low Self-Esteem                  0.000   0.000   0.000      111        0
Concentration Problems           0.000   0.000   0.000       72        0
Sleeping Disorder                0.000   0.000   0.000       78        0

✓ Metrics saved!

=== COMPARISON TO BASELINE ===
InstructBLIP (zero-shot):           Macro-F1 = 0.25 (25%)
BLIP-2 (explanation-aug

In [9]:
# Quick test: BLIP-2 without explanations
def simple_prompt():
    return (
        "List all mental health issues shown in this meme. "
        "Options: depression, loss of interest, self-harm, eating problems, "
        "low self-esteem, concentration issues, sleep problems. "
        "List all that apply, separated by commas."
    )

print("=== TESTING BLIP-2 WITHOUT EXPLANATIONS ===\n")

# Test on 10 images
for i in range(10):
    row = metadata_df.iloc[i]
    image = Image.open(DATA_DIR / row['image_path']).convert('RGB')
    prompt = simple_prompt()

    inputs = processor(images=image, text=prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7)

    response = processor.batch_decode(out, skip_special_tokens=True)[0]

    print(f"Image {i+1}: {row['image_id']}")
    print(f"Response: '{response}'")
    print("-" * 80)

=== TESTING BLIP-2 WITHOUT EXPLANATIONS ===

Image 1: TE-104.jpg
Response: 'For most. This doesnâ  feel okay without an adjective associated only directly and can – like'
--------------------------------------------------------------------------------
Image 2: TE-177.jpg
Response: 'Insecle is one step away into one side on another cell y. anxiety also the same emotion of being angry in.bxreportin anger repress hatred sadness lack. it anger/everything feelings for feelings etcs are feelings you cannot see how it has led here or the relationship just because... of seeing the abuse online may only in online be directed there means like there the thing . its own emotion just because no cause at an angle like face for or.'
--------------------------------------------------------------------------------
Image 3: TE-184.jpg
Response: 'depraving themselves leads with these negative associations is'
--------------------------------------------------------------------------------
Image 4: TE-195

In [10]:
print("=== TESTING BLIP-2 WITH GREEDY DECODING ===\n")

# Test on 5 images with greedy (no sampling)
for i in range(5):
    row = metadata_df.iloc[i]
    image = Image.open(DATA_DIR / row['image_path']).convert('RGB')
    prompt = simple_prompt()

    inputs = processor(images=image, text=prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=50,  # Shorter
            do_sample=False,     # GREEDY - no sampling
            num_beams=1
        )

    response = processor.batch_decode(out, skip_special_tokens=True)[0]

    print(f"Image {i+1}: {row['image_id']}")
    print(f"Response: '{response}'")
    print(f"Expected: {row['human_label']}")
    print("-" * 80)

=== TESTING BLIP-2 WITH GREEDY DECODING ===

Image 1: TE-104.jpg
Response: 'i am not interested in you'
Expected: [1,0,0,0,0,0,0]
--------------------------------------------------------------------------------
Image 2: TE-177.jpg
Response: 'i am not interested in this movie'
Expected: [1,0,0,0,1,0,0]
--------------------------------------------------------------------------------
Image 3: TE-184.jpg
Response: 'i am not sure what you mean'
Expected: [1,0,1,0,0,0,0]
--------------------------------------------------------------------------------
Image 4: TE-195.jpg
Response: 'i am not sure what you mean by saying i am not sure what you mean by saying i am not sure what you mean by saying i am not sure what you mean by saying i am not sure what you mean by saying'
Expected: [0,0,0,1,1,0,0]
--------------------------------------------------------------------------------
Image 5: TE-285.jpg
Response: 'i am not sure what you mean'
Expected: [1,0,1,0,0,0,0]
----------------------------------

In [11]:
# Save BLIP-2 failure summary
summary = """
# EXPERIMENT 3 - BLIP-2 EXPLANATION-AUGMENTED (FAILED)

## Date: 2026-04-01

## Model Configuration
- Model: Salesforce/blip2-flan-t5-xl
- GPU: NVIDIA A100-SXM4-40GB
- Approach: Explanation + Image (natural language prompts)

## Results
**CATASTROPHIC FAILURE**
- Macro-F1: 0.0126 (1.3%)
- 95% WORSE than InstructBLIP baseline (25%)

## Per-Symptom Performance
- Feeling Down: 8.8% F1 (only symptom detected at all)
- All other symptoms: 0.0% F1 (zero detections)

## What Went Wrong

### Issue 1: Model produces incoherent outputs
Sample outputs:
- "social addiction-an individual committing activity/service with others..."
- "neurochemical level imbalance during periods known but without disturbance..."
- "no loss about passion can solve my majoring of psychology at home"

### Issue 2: Even without explanations, model fails
Tested image-only prompts (no explanations):
- Still produces gibberish
- Generic conversational responses ("I am not interested in you")
- Repetition loops ("I am not sure what you mean by saying...")

### Issue 3: Greedy decoding doesn't help
- Tried deterministic generation (no sampling)
- Still produces generic conversation, not symptom lists
- Model fundamentally cannot follow structured instructions

## Root Cause
BLIP-2-Flan-T5 is trained for:
- Open-ended visual question answering
- Conversational dialogue
- Image captioning

NOT for:
- Structured classification
- Multi-label symptom detection
- Following specific output formats

## Comparison to InstructBLIP
InstructBLIP (25% F1) vs BLIP-2 (1.3% F1):
- InstructBLIP: Designed for instruction-following
- BLIP-2: Designed for open conversation
- Task requires instruction-following capability

## Research Implications
- Not all vision-language models are suitable for classification tasks
- Instruction-tuned models (InstructBLIP) > Conversational models (BLIP-2)
- Explanation-augmentation ONLY helps if base model works
- Baseline model capability is prerequisite for prompting strategies

## Files Generated
- blip2_explanation_augmented.csv (650 predictions - mostly [0,0,0,0,0,0,0])
- blip2_explanation_augmented_metrics.csv (1.3% macro-F1)

## Recommendation
- DO NOT use BLIP-2 for this task
- InstructBLIP baseline (25%) remains best result
- For Track B (explanation-augmented), need different model:
  - LLaVA-NeXT-7B (proven instruction-following)
  - Or proceed directly to Track C/D (fine-tuning)
"""

# Save
summary_path = RESULTS_DIR / 'experiment_3_blip2_failure_summary.txt'
with open(summary_path, 'w') as f:
    f.write(summary)

print(summary)
print(f"\n✓ Summary saved to: {summary_path}")

# Download
from google.colab import files
files.download(str(summary_path))

print("\n✓ Download initiated")


# EXPERIMENT 3 - BLIP-2 EXPLANATION-AUGMENTED (FAILED)

## Date: 2026-04-01

## Model Configuration
- Model: Salesforce/blip2-flan-t5-xl
- GPU: NVIDIA A100-SXM4-40GB
- Approach: Explanation + Image (natural language prompts)

## Results
**CATASTROPHIC FAILURE**
- Macro-F1: 0.0126 (1.3%)
- 95% WORSE than InstructBLIP baseline (25%)

## Per-Symptom Performance
- Feeling Down: 8.8% F1 (only symptom detected at all)
- All other symptoms: 0.0% F1 (zero detections)

## What Went Wrong

### Issue 1: Model produces incoherent outputs
Sample outputs:
- "social addiction-an individual committing activity/service with others..."
- "neurochemical level imbalance during periods known but without disturbance..."
- "no loss about passion can solve my majoring of psychology at home"

### Issue 2: Even without explanations, model fails
Tested image-only prompts (no explanations):
- Still produces gibberish
- Generic conversational responses ("I am not interested in you")
- Repetition loops ("I am not 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ Download initiated


In [13]:
import os

print("=== FINAL VERIFICATION ===\n")
print(f"Results directory: {RESULTS_DIR}\n")

# Check all result files
files_to_check = [
    'instructblip_natural_language.csv',
    'instructblip_natural_language_metrics.csv',
    'instructblip_natural_language_metrics.json',
    'experiment_1_summary.txt',
    'palo7b_attempt_summary.txt',
    'blip2_explanation_augmented.csv',
    'blip2_explanation_augmented_metrics.csv',
    'blip2_explanation_augmented_metrics.json',
    'experiment_3_blip2_failure_summary.txt'
]

print("Checking saved files:")
for filename in files_to_check:
    filepath = RESULTS_DIR / filename
    if filepath.exists():
        size = filepath.stat().st_size / 1024
        print(f"  ✅ {filename} ({size:.1f} KB)")
    else:
        print(f"  ❌ MISSING: {filename}")

print("\n" + "=" * 80)
print("EXPERIMENT SUMMARY")
print("=" * 80)
print("\n✅ EXPERIMENT 1: InstructBLIP")
print("   - Status: SUCCESS")
print("   - Macro-F1: 25%")
print("   - Files: 3")

print("\n❌ EXPERIMENT 2: PALO-7B")
print("   - Status: FAILED (model loading issue)")
print("   - Files: 1 (attempt summary)")

print("\n❌ EXPERIMENT 3: BLIP-2")
print("   - Status: FAILED (1.3% F1)")
print("   - Files: 3")

print("\n" + "=" * 80)
print("✅ ALL FILES SAVED TO GOOGLE DRIVE")
print("=" * 80)

=== FINAL VERIFICATION ===

Results directory: /content/drive/MyDrive/colab_experiments/results

Checking saved files:
  ✅ instructblip_natural_language.csv (76.0 KB)
  ✅ instructblip_natural_language_metrics.csv (0.6 KB)
  ✅ instructblip_natural_language_metrics.json (1.6 KB)
  ✅ experiment_1_summary.txt (1.7 KB)
  ✅ palo7b_attempt_summary.txt (3.0 KB)
  ✅ blip2_explanation_augmented.csv (149.7 KB)
  ✅ blip2_explanation_augmented_metrics.csv (0.3 KB)
  ✅ blip2_explanation_augmented_metrics.json (1.3 KB)
  ✅ experiment_3_blip2_failure_summary.txt (2.4 KB)

EXPERIMENT SUMMARY

✅ EXPERIMENT 1: InstructBLIP
   - Status: SUCCESS
   - Macro-F1: 25%
   - Files: 3

❌ EXPERIMENT 2: PALO-7B
   - Status: FAILED (model loading issue)
   - Files: 1 (attempt summary)

❌ EXPERIMENT 3: BLIP-2
   - Status: FAILED (1.3% F1)
   - Files: 3

✅ ALL FILES SAVED TO GOOGLE DRIVE
